# Classical Machine Learning: a Bridge in One Notebook

Companion notebook to *Build AI, Understand AI*, Part VI, Chapter 1
(*The Older AI in the Room*).

Before language models, and still running underneath much of the world,
sits classical machine learning: models fitted to tables of numbers.
In one short sitting and with no model server at all, this notebook:

1. trains a **decision tree** to classify iris flowers from four measurements;
2. reads the learned rules back as plain if/else statements;
3. measures it honestly on data it never trained on;
4. throws away the labels and lets **k-means** find groups on its own.

The workflow in every case is `fit`, `predict`, `evaluate`, which is the
classical ancestor of the training loop in `train_tiny_model.ipynb`.
Needs only scikit-learn (already a hands-on-ai dependency).


## Step 0: Settings

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score, confusion_matrix

SEED = 42


## Step 1: A table, features, and a label

The iris dataset ships inside scikit-learn, so nothing is downloaded.
Each of 150 flowers is one row: four measurements (features) and a
species number (the label a classifier must predict). Rows with labels
are supervised learning: the answer key exists, and part of it will be
sealed away for testing, the discipline from the training chapters.


In [2]:
iris = load_iris()
X, y = iris.data, iris.target
print("rows:", X.shape, "| features:", list(iris.feature_names))
print("classes:", list(iris.target_names), "| counts:", {n: int((y == i).sum()) for i, n in enumerate(iris.target_names)})
print("first row:", X[0], "->", iris.target_names[y[0]])


rows: (150, 4) | features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')] | counts: {np.str_('setosa'): 50, np.str_('versicolor'): 50, np.str_('virginica'): 50}
first row: [5.1 3.5 1.4 0.2] -> setosa


## Step 2: Seal a test set, then fit

Thirty percent of rows, stratified so each species keeps its share, are put aside before any fitting. The tree will never see them until scoring.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
print(f"train: {len(X_train)} rows | sealed test: {len(X_test)} rows")

tree = DecisionTreeClassifier(random_state=SEED)
tree.fit(X_train, y_train)
print("fitted: depth", tree.get_depth(), "| leaves", tree.get_n_leaves())


train: 105 rows | sealed test: 45 rows
fitted: depth 5 | leaves 8


## Step 3: Read the model back

A decision tree is the rare model you can print and read on one screen.
These rules *are* the model: no weights soup, no prompt, just thresholds
on measurements, chosen because they separated the training rows well.


In [4]:
print(export_text(tree, feature_names=list(iris.feature_names), max_depth=3))


|--- petal length (cm) <= 2.45
|   |--- class: 0
|--- petal length (cm) >  2.45
|   |--- petal width (cm) <= 1.55
|   |   |--- petal length (cm) <= 4.95
|   |   |   |--- class: 1
|   |   |--- petal length (cm) >  4.95
|   |   |   |--- class: 2
|   |--- petal width (cm) >  1.55
|   |   |--- petal width (cm) <= 1.70
|   |   |   |--- sepal width (cm) <= 2.85
|   |   |   |   |--- class: 1
|   |   |   |--- sepal width (cm) >  2.85
|   |   |   |   |--- class: 2
|   |   |--- petal width (cm) >  1.70
|   |   |   |--- petal length (cm) <= 4.85
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- petal length (cm) >  4.85
|   |   |   |   |--- class: 2



## Step 4: Score on data it never saw

In [5]:
pred = tree.predict(X_test)
print("held-out accuracy:", round(accuracy_score(y_test, pred), 3))
print("confusion matrix (rows=true, cols=predicted):\n", confusion_matrix(y_test, pred))
wrong = [i for i, (a, b) in enumerate(zip(y_test, pred)) if a != b]
print("mistakes:", len(wrong), "of", len(y_test))
for i in wrong:
    print(" features", X_test[i], "true", iris.target_names[y_test[i]], "said", iris.target_names[pred[i]])


held-out accuracy: 0.933
confusion matrix (rows=true, cols=predicted):
 [[15  0  0]
 [ 0 12  3]
 [ 0  0 15]]
mistakes: 3 of 45
 features [6.3 3.3 4.7 1.6] true versicolor said virginica
 features [6.  3.4 4.5 1.6] true versicolor said virginica
 features [6.7 3.  5.  1.7] true versicolor said virginica


## Step 5: No labels at all (k-means)

Clustering asks a different kind of question: ignoring species entirely,
can geometry alone find groups in the measurements? K-means picks three
centres and repeatedly moves them to the middle of the points nearest
them. Nobody told it what a species is. Comparing its groups with the
true labels afterwards (adjusted Rand index: 1.0 is perfect agreement,
0 is chance) shows how much structure was in the numbers alone.


In [6]:
km = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(X)
print("ARI vs true species:", round(adjusted_rand_score(y, km.labels_), 3))
import collections
for c in range(3):
    members = y[km.labels_ == c]
    mix = {iris.target_names[k]: int(v) for k, v in collections.Counter(members).items()}
    print(f"cluster {c}: {mix}")


ARI vs true species: 0.73
cluster 0: {np.str_('versicolor'): 48, np.str_('virginica'): 14}
cluster 1: {np.str_('setosa'): 50}
cluster 2: {np.str_('versicolor'): 2, np.str_('virginica'): 36}


## What to notice before moving on

- The tree needed no server, no prompt, and no tokens; fitting took
  milliseconds and the result is inspectable line by line.
- Inputs were numbers in fixed slots. Raw text, images, and audio do
  not arrive that way, which is exactly the gap deep learning, and
  then LLMs, filled, at enormous cost in data, compute, and opacity.
- The evaluation habits carried over unchanged: seal test data before
  fitting, read your mistakes individually, distrust training-set
  scores. The next notebook (`train_tiny_model.ipynb`) runs the same
  discipline with gradient descent in place of rule thresholds.

Try next: limit the tree to `max_depth=2` and watch accuracy trade
against readability; run k-means with the wrong `n_clusters`; swap the
tree for `SVC` (a support vector machine) and compare held-out scores
on identical splits.
